# Understanding Convolutional Neural Network Operations
This notebook demonstrates the fundamental operations used in CNNs:

* convolution,
* activation functions,
* pooling.

The code uses explicit loops for clarity rather than vectorized operations, making the implementation easier to understand for educational purposes.

Note that in practice, deep learning frameworks use optimized implementations for better performance.

In [2]:
import numpy as np

## 1. Convolution Operation
The convolution operation applies filters to extract features from input data.

Key parameters:
- Padding: Adds zeros around the input
- Stride: Step size for filter movement
- Dilation: Spacing between filter elements

In [5]:
def convolution2d(input, kernel, padding=0, stride=1, dilation=1):
    """
    Performs 2D convolution with padding, stride, and dilation.
    Handles both 2D (grayscale) and 3D (color) inputs.
    Input shape: (height, width) or (height, width, channels)
    Kernel shape: (size, size) or (size, size, channels)
    """

    # Check input dimensions and adjust padding accordingly
    if input.ndim == 2:  # Grayscale input
        input_padded = np.pad(input, padding, mode='constant')
        kernel_channels = 1  # Single channel for grayscale kernel
    elif input.ndim == 3:  # Color input
        input_padded = np.pad(input, ((padding, padding), (padding, padding), (0, 0)), mode='constant')
        kernel_channels = kernel.shape[-1]  # Get channels from kernel shape
    else:
        raise ValueError("Input must be 2D (grayscale) or 3D (color)")

    # Get dimensions
    n, m = input_padded.shape[:2]  # Get height and width of padded input
    k = kernel.shape[0]  # Assuming kernel is square

    # Calculate output dimensions
    output_h = (n - k + 2 * padding) // stride + 1
    output_w = (m - k + 2 * padding) // stride + 1

    output = np.zeros((output_h, output_w))

    # Perform convolution
    for i in range(output_h):
        for j in range(output_w):
            # Calculate window position
            h_start = i * stride
            h_end = h_start + k
            w_start = j * stride
            w_end = w_start + k

            # Extract window from padded input
            input_window = input_padded[h_start:h_end, w_start:w_end]

            # Apply kernel to window and accumulate result
            if input.ndim == 2:
                output[i, j] = np.sum(input_window * kernel)
            else:  # Color input
                for c in range(kernel_channels):
                    output[i, j] += np.sum(input_window[..., c] * kernel[..., c])

    return output

## 2. Activation Function
Activation functions introduce non-linearity. Common choices:
- ReLU: Returns max(0, x)
- Sigmoid: Squashes values between 0 and 1

In [8]:
def activation(input, activation_type='relu'):
    """Applies specified activation function"""
    if activation_type.lower() == 'relu':
        return np.maximum(0, input)
    elif activation_type.lower() == 'sigmoid':
        return 1 / (1 + np.exp(-input))
    else:
        raise ValueError("Unsupported activation function")

## 3. Pooling Operation
Pooling reduces spatial dimensions while preserving important features:
- Max pooling: Takes maximum value in window
- Average pooling: Takes mean value in window

In [11]:
def pooling(input, pool_size=2, stride=2, pool_type='max'):
    """Performs pooling operation on 2D input"""
    h, w = input.shape
    output_h = (h - pool_size) // stride + 1
    output_w = (w - pool_size) // stride + 1

    output = np.zeros((output_h, output_w))

    for i in range(output_h):
        for j in range(output_w):
            h_start = i * stride
            h_end = h_start + pool_size
            w_start = j * stride
            w_end = w_start + pool_size

            window = input[h_start:h_end, w_start:w_end]

            if pool_type.lower() == 'max':
                output[i, j] = np.max(window)
            elif pool_type.lower() == 'mean':
                output[i, j] = np.mean(window)
            else:
                raise ValueError("Unsupported pool type")

    return output

## Example Usage
Let's test our functions with sample input:

\begin{bmatrix}
1 & 0 & 0 & 1 & -1 \\
2 & 1 & 1 & 0 & 0 \\
0 & -1 & 0 & 1 & 2 \\
-1 & -1 & 1 & 1 & -1 \\
0 & 1 & 0 & 1 & 0
\end{bmatrix}

In [29]:
# Create sample input (5x5)
input = np.array([[1,0,0,1,-1],
                  [2,1,1,0,0],
                  [0,-1,0,1,2],
                  [-1,-1,1,1,-1],
                  [0,1,0,1,0]])

input = np.array([[0,0,0,0,0],
                  [0,0,0,0,0],
                  [0,0,0,0,0],
                  [0,0,0,0,0],
                  [0,0,0,0,0]])

input = np.array([[0,0,0,0,0,0,0,0,0],
                  [0,1,1,0,0,0,0,0,0],
                  [0,1,1,0,0,0,0,0,0],
                  [0,1,1,0,0,1,1,0,0],
                  [0,1,1,1,1,1,1,0,0],
                  [0,1,1,1,1,1,1,0,0],
                  [0,0,0,0,0,1,1,0,0],
                  [0,0,0,0,0,1,1,0,0],
                  [0,0,0,0,0,0,0,0,0]])

# Number of channel
c = 3
# Create sample input (5x5xc)
#input = np.random.randn(5, 5, c)

# Create 3x3 kernel
kernel = np.array([[1,0,1], [0,1,0], [1,0,1]])
kernel = np.array([[-1,0,1], [-2,0,2], [-1,0,1]])
#kernel = np.random.randn(3, 3, c)

# Perform convolution
conv_result = convolution2d(input, kernel, padding=0, stride=2)
# Apply ReLU activation
activated = activation(conv_result)
# Perform max pooling
pooled = pooling(activated, pool_size=2, stride=1)

print(f"Input shape: {input.shape}")
print(f"Convolution output shape: {conv_result.shape}")
print(f"Activation output shape: {activated.shape}")
print(f"Pooling output shape: {pooled.shape}")

Input shape: (9, 9)
Convolution output shape: (4, 4)
Activation output shape: (4, 4)
Pooling output shape: (3, 3)


In [31]:
print(f"Input:\n{input}")
print(f"Kernel:\n{kernel}")
print(f"Convolution output:\n{conv_result}")
print(f"Activation output:\n{activated}")
print(f"Pooling output:\n{pooled}")

Input:
[[0 0 0 0 0 0 0 0 0]
 [0 1 1 0 0 0 0 0 0]
 [0 1 1 0 0 0 0 0 0]
 [0 1 1 0 0 1 1 0 0]
 [0 1 1 1 1 1 1 0 0]
 [0 1 1 1 1 1 1 0 0]
 [0 0 0 0 0 1 1 0 0]
 [0 0 0 0 0 1 1 0 0]
 [0 0 0 0 0 0 0 0 0]]
Kernel:
[[-1  0  1]
 [-2  0  2]
 [-1  0  1]]
Convolution output:
[[ 3. -3.  0.  0.]
 [ 4. -3.  2. -3.]
 [ 3.  0.  1. -4.]
 [ 0.  0.  3. -3.]]
Activation output:
[[3. 0. 0. 0.]
 [4. 0. 2. 0.]
 [3. 0. 1. 0.]
 [0. 0. 3. 0.]]
Pooling output:
[[4. 2. 2.]
 [4. 2. 2.]
 [3. 3. 3.]]
